# 18 — Campaigns

**The concept:** a study is rarely one run. It is a grid of cases, each repeated over random
realizations: many points of one pipeline. A **campaign** runs them in parallel in the project's
own environment, quotes its cost from one point before starting, records every point as it
finishes, and **resumes from where it stopped, whatever stopped it**: a budget, a stop request,
the session ending, the process dying.

Three rules govern it:

1. **A number before a commitment.** One point is run and measured first, and starting needs a
   budget. There is no default.
2. **Failures are recorded, never dropped.** An average that silently lost its failures would
   present itself as a complete measurement.
3. **Two versions of a model are never averaged together.**

In [1]:
import pathlib, tempfile, time
from smartmdao.sweep import Campaign

workdir = pathlib.Path(tempfile.mkdtemp())
model = workdir / "model.py"
model.write_text(
    "import random, time\n"
    "from smartmdao import Pipeline\n"
    "pipeline = Pipeline(inputs=['scale', 'seed'])\n"
    "@pipeline.step(outputs=['value'])\n"
    "def noisy(scale: float, seed: int) -> float:\n"
    "    if scale < 0:\n"
    "        raise ValueError('the solver reported: inaccurate')\n"
    "    time.sleep(0.05)\n"
    "    return scale * random.Random(seed).random()\n"
)

campaign = Campaign(model, store=workdir / "store", grid={"scale": [1.0, 2.0]},
                    seeds=range(10), seed_input="seed", outputs=["value"], workers=2)
print("points:", len(campaign.design()), "- 2 cases x 10 realizations")

points: 20 - 2 cases x 10 realizations


**Seeds are ordinary inputs.** The model takes its randomness from `seed`; SmartMDAO never
seeds a global random state. A campaign whose seed no step reads is refused: its
"realizations" would all be the same run.

## The quote comes first

One point is run in the project's environment and measured, and the rest is estimated from it.
The point is recorded like any other, so it is never computed twice:

In [2]:
quote = campaign.quote()
print("pending after the quote:", quote["pending"], "on", quote["workers"], "workers")
print("the measured point:", quote["point"]["status"])
print(quote["note"])

pending after the quote: 19 on 2 workers
the measured point: ok
One point took 0.050554s. The 19 left on 2 worker(s) should take about 1.1s. Quote this before starting; starting needs a budget.


## Stopped part-way, then resumed

A budget too small for the whole campaign stops it. The points already finished are on disk:
each is appended to `points.jsonl` and flushed before the next is handed out. Running the same
campaign again does **only what is missing**:

In [3]:
stopped = campaign.run(budget_seconds=1.0)
print("first run: ", stopped["run"]["state"], "| some points pending:", stopped["counts"].get("pending", 0) > 0)

finished = campaign.run(budget_seconds=120)
print("second run:", finished["run"]["state"], finished["counts"])

records = (workdir / "store" / "points.jsonl").read_text().splitlines()
print("records written:", len(records), "for", len(campaign.design()), "points - none computed twice")

first run:  stopped: budget | some points pending: True


second run: finished {'ok': 20}
records written: 20 for 20 points - none computed twice


The same holds when the session ends or the process is killed outright: `start()` runs the
campaign in its own process, and starting it again resumes it. The test suite does exactly that,
with `kill -9`. The store holds only each point's description and its outputs, **never its
inputs**, which the project's code rebuilds when a point runs. So no input type, whether string,
tuple or array, stands in the way of a resume.

## What the numbers are, and what they left out

In [4]:
for group in campaign.aggregate()["groups"]:
    value = group["outputs"]["value"]
    print(f"scale={group['params']['scale']}: n={value['n']}, mean={value['mean']:.3f}, "
          f"95% interval ±{value['ci_half_width']:.3f}, excluded={group['excluded']}")

scale=1.0: n=10, mean=0.484, 95% interval ±0.214, excluded={}
scale=2.0: n=10, mean=0.968, 95% interval ±0.428, excluded={}


Now a campaign where some points fail. Every point ends in exactly one status: `ok`,
`not_converged`, `error`, `timed_out` or `crashed`. The failures are listed, and the aggregate
counts them beside the numbers instead of quietly averaging without them:

In [5]:
mixed = Campaign(model, store=workdir / "mixed", grid={"scale": [1.0, -1.0]},
                 seeds=range(3), seed_input="seed", outputs=["value"], workers=2)
status = mixed.run(budget_seconds=120)
print("counts:", status["counts"])
print("a failure:", status["failures"][0]["error"])
for group in mixed.aggregate()["groups"]:
    print(f"scale={group['params']['scale']}: numbers from", group["outputs"].get("value", {}).get("n", 0),
          "points, excluded", group["excluded"])

counts: {'ok': 3, 'error': 3}
a failure: RuntimeError: Error executing step 'noisy': the solver reported: inaccurate
scale=-1.0: numbers from 0 points, excluded {'error': 3}
scale=1.0: numbers from 3 points, excluded {}


## An edited model starts afresh

A point's key includes a hash of the model's source and of the SmartMDAO version that ran it. Edit
the model and nothing stored matches: its old points are **set aside**, kept but never averaged
with the new ones.

In [6]:
model.write_text(model.read_text().replace("scale * random", "2 * scale * random"))
again = campaign.run(budget_seconds=120)
print("newly computed:", again["run"]["finished"], "| set aside:", again["set_aside"])
print(again["note"])

newly computed: 20 | set aside: 20
20 stored point(s) belong to another version of the model or of SmartMDAO. They are kept but never mixed in; their points run again.


## Detached

`start()` returns at once. The campaign runs in its own process, which outlives the session that
started it, and `status()` reads its progress from the store. A heartbeat tells *running* from
*interrupted*. Over MCP the same is `sweep_pipeline(..., start=True, budget_seconds=...)`,
`sweep_status` and `sweep_stop`.

In [7]:
detached = Campaign(model, store=workdir / "detached", grid={"scale": [3.0]},
                    seeds=range(6), seed_input="seed", outputs=["value"], workers=2)
print("started:", detached.start(budget_seconds=120)["started"])
while detached.status()["state"] in ("starting", "running"):
    time.sleep(0.5)
print("then:", detached.status()["state"], detached.status()["counts"])

started: True


then: finished {'ok': 6}


In [8]:
import shutil
shutil.rmtree(workdir)
print("cleaned up")

cleaned up


---

Why it is built this way, including the measurement behind the worker pool (a small point cost
0.71 s through a fresh process and 0.66 ms in a warm one), is
[design record 006](../docs/design/006-sweep.md).

**Back to** [the index](README.md).